In [26]:


import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

# Models
from xgboost import XGBClassifier

# Preprocessing & splitting
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score,
    precision_score, recall_score,
    RocCurveDisplay
)



## 2. Load Data

In [2]:
orders               = pd.read_csv("orders.csv")
products             = pd.read_csv("products.csv")
order_products_prior = pd.read_csv("order_products__prior.csv")
order_products_train = pd.read_csv("order_products__train.csv")

print("orders:              ", orders.shape)
print("products:            ", products.shape)
print("order_products_prior:", order_products_prior.shape)
print("order_products_train:", order_products_train.shape)

orders:               (3421083, 7)
products:             (49688, 4)
order_products_prior: (32434489, 4)
order_products_train: (1384617, 4)


## 3. Build Prior Data

In [3]:
prior_orders = orders[orders["eval_set"] == "prior"].copy()
prior_data   = prior_orders.merge(order_products_prior, on="order_id", how="inner")

print("prior_data shape:", prior_data.shape)
prior_data.head(3)

prior_data shape: (32434489, 10)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,2539329,1,prior,1,2,8,NaN,196,1,0
1,2539329,1,prior,1,2,8,NaN,14084,2,0
2,2539329,1,prior,1,2,8,NaN,12427,3,0


## 4. Feature Engineering
Build 19 features across 3 groups: user-product, user-level, product-level.

In [4]:
# ── USER-PRODUCT FEATURES ─────────────────────────────────────────────────────
# (up means - user-product)

# UP1: times user bought this product
up_count = (
    prior_data.groupby(["user_id", "product_id"])
    .size().reset_index(name="up_times_bought")
)

# UP2: last order number when user bought this product
up_last = (
    prior_data.groupby(["user_id", "product_id"])["order_number"]
    .max().reset_index(name="up_last_order")
)

# UP3: first order number when user bought this product
up_first = (
    prior_data.groupby(["user_id", "product_id"])["order_number"]
    .min().reset_index(name="up_first_order")
)

# UP4: avg add-to-cart position for this user-product pair
up_cart_pos = (
    prior_data.groupby(["user_id", "product_id"])["add_to_cart_order"]
    .mean().reset_index(name="up_avg_cart_pos")
)

print("User-product features: done")

User-product features: done


In [5]:
# ── USER-LEVEL FEATURES ────────────────────────────────────────────────────────

# U1: total orders
u_total_orders = (
    prior_orders.groupby("user_id")["order_number"]
    .max().reset_index(name="u_total_orders")
)

# U2: total items bought across all prior orders
u_total_items = (
    prior_data.groupby("user_id")
    .size().reset_index(name="u_total_items")
)

# U3: average days between orders
u_avg_days = (
    prior_orders.groupby("user_id")["days_since_prior_order"]
    .mean().reset_index(name="u_avg_days_between_orders")
)

# U4: average basket size
u_basket_size = (
    prior_data.groupby(["user_id", "order_id"])
    .size().reset_index(name="basket_size")
    .groupby("user_id")["basket_size"]
    .mean().reset_index(name="u_avg_basket_size")
)

# U5: unique products ever bought
u_unique_products = (
    prior_data.groupby("user_id")["product_id"]
    .nunique().reset_index(name="u_unique_products")
)

# U6: user reorder ratio (how often they reorder vs buy new)
u_reorder_ratio = (
    prior_data.groupby("user_id")["reordered"]
    .mean().reset_index(name="u_reorder_ratio")
)

print("User-level features: done")

User-level features: done


In [6]:
# ── PRODUCT-LEVEL FEATURES ────────────────────────────────────────────────────

# P1: product total purchase count
p_purchase_count = (
    prior_data.groupby("product_id")
    .size().reset_index(name="p_purchase_count")
)

# P2: product reorder rate
p_reorder_rate = (
    prior_data.groupby("product_id")["reordered"]
    .mean().reset_index()
    .rename(columns={"reordered": "p_reorder_rate"})
)

# P3: product unique buyers
p_unique_buyers = (
    prior_data.groupby("product_id")["user_id"]
    .nunique().reset_index(name="p_unique_buyers")
)

# P4: avg add-to-cart position across all orders
p_avg_cart_pos = (
    prior_data.groupby("product_id")["add_to_cart_order"]
    .mean().reset_index(name="p_avg_cart_pos")
)

print("Product-level features: done")

Product-level features: done


## 5. Merge All Features

In [7]:
features = (
    up_count
    .merge(up_last,          on=["user_id", "product_id"], how="left")
    .merge(up_first,         on=["user_id", "product_id"], how="left")
    .merge(up_cart_pos,      on=["user_id", "product_id"], how="left")
    .merge(u_total_orders,   on="user_id", how="left")
    .merge(u_total_items,    on="user_id", how="left")
    .merge(u_avg_days,       on="user_id", how="left")
    .merge(u_basket_size,    on="user_id", how="left")
    .merge(u_unique_products,on="user_id", how="left")
    .merge(u_reorder_ratio,  on="user_id", how="left")
    .merge(p_purchase_count, on="product_id", how="left")
    .merge(p_reorder_rate,   on="product_id", how="left")
    .merge(p_unique_buyers,  on="product_id", how="left")
    .merge(p_avg_cart_pos,   on="product_id", how="left")
)

# ── DERIVED FEATURES ──────────────────────────────────────────────────────────

# How often user buys this product per order (0–1)
features["up_purchase_rate"]    = features["up_times_bought"] / features["u_total_orders"]

# Orders since user last bought this product (recency — lower = more likely to reorder)
features["up_orders_since_last"] = features["u_total_orders"] - features["up_last_order"]

# Share of product in user's total history
features["up_share"]            = features["up_times_bought"] / features["u_total_items"]

# How consistently they buy it over time (density between first and last purchase)
features["up_reorder_density"]  = features["up_times_bought"] / (
    features["up_last_order"] - features["up_first_order"] + 1
)

# User shopping diversity (more diverse = less likely to reorder any single item)
features["u_diversity"]         = features["u_unique_products"] / features["u_total_items"]

# Product repeat loyalty (avg times a buyer comes back for it)
features["p_loyalty"]           = features["p_purchase_count"] / features["p_unique_buyers"]

# Fill NaN from days_since_prior_order (first orders)
features["u_avg_days_between_orders"] = features["u_avg_days_between_orders"].fillna(0)

print("features shape:", features.shape)
print("Columns:", features.columns.tolist())

features shape: (13307953, 22)
Columns: ['user_id', 'product_id', 'up_times_bought', 'up_last_order', 'up_first_order', 'up_avg_cart_pos', 'u_total_orders', 'u_total_items', 'u_avg_days_between_orders', 'u_avg_basket_size', 'u_unique_products', 'u_reorder_ratio', 'p_purchase_count', 'p_reorder_rate', 'p_unique_buyers', 'p_avg_cart_pos', 'up_purchase_rate', 'up_orders_since_last', 'up_share', 'up_reorder_density', 'u_diversity', 'p_loyalty']


## 6. Build Train Labels

In [8]:
train_orders = orders[orders["eval_set"] == "train"].copy()
train_data   = train_orders.merge(order_products_train, on="order_id", how="left")

train_labels            = train_data[["user_id", "product_id"]].copy()
train_labels["target"]  = 1

print("train_labels shape:", train_labels.shape)

train_labels shape: (1384617, 3)


## 7. Build Final DataFrame (KEY FIX: filter to train users only)

In [9]:
# ── CRITICAL FIX ──────────────────────────────────────────────────────────────
# Only keep user-product pairs for users who appear in the train set.
# Without this, millions of prior-only users flood class 0 and kill recall.
train_user_ids = train_orders["user_id"].unique()
features_train = features[features["user_id"].isin(train_user_ids)].copy()

# Attach labels
final_df = features_train.merge(train_labels, on=["user_id", "product_id"], how="left")
final_df["target"] = final_df["target"].fillna(0).astype("int8")

print("final_df shape:", final_df.shape)
print("\nTarget distribution:")
print(final_df["target"].value_counts())
print(final_df["target"].value_counts(normalize=True).round(3))

final_df shape: (8474661, 23)

Target distribution:
target
0    7645837
1     828824
Name: count, dtype: int64
target
0    0.902
1    0.098
Name: proportion, dtype: float64


In [10]:
# Sanity check: reordered products should have clearly higher values
final_df.groupby("target")[
    ["up_times_bought", "up_purchase_rate", "up_orders_since_last",
     "up_reorder_density", "p_reorder_rate"]
].mean().round(3)

,up_times_bought,up_purchase_rate,up_orders_since_last,up_reorder_density,p_reorder_rate
target,,,,,
0,2.145,0.134,10.301,0.814,0.526
1,5.113,0.334,2.323,0.736,0.617


In [29]:


FEATURE_COLS = [
    # User-product
    "up_times_bought", "up_last_order", "up_first_order", "up_avg_cart_pos",
    "up_purchase_rate", "up_orders_since_last", "up_share", "up_reorder_density",

    # User
    "u_total_orders", "u_total_items", "u_avg_days_between_orders",
    "u_avg_basket_size", "u_unique_products", "u_reorder_ratio", "u_diversity",

    # Product
    "p_purchase_count", "p_reorder_rate", "p_unique_buyers",
    "p_avg_cart_pos", "p_loyalty"
]

# split based on final_df["target"]
split = StratifiedShuffleSplit(
    n_splits=1,
    train_size=1_000_000,
    random_state=42
)

for _,sample_idx in split.split(final_df, final_df["target"]):
    sampled_df = final_df.iloc[sample_idx].copy()

X = sampled_df[FEATURE_COLS].copy()
y = sampled_df["target"].copy()

print("X shape:", X.shape)
print("Class balance after filter:")
print(y.value_counts(normalize=True).round(3))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))
print("Test class balance:\n", y_test.value_counts(normalize=True).round(3))

X shape: (1000000, 20)
Class balance after filter:
target
0    0.902
1    0.098
Name: proportion, dtype: float64
X_train: (800000, 20) | X_test: (200000, 20)
Train class balance:
 target
0    0.902
1    0.098
Name: proportion, dtype: float64
Test class balance:
 target
0    0.902
1    0.098
Name: proportion, dtype: float64


## 8. Prepare X, y

In [30]:
# FEATURE_COLS = [
#     # User-product
#     "up_times_bought", "up_last_order", "up_first_order", "up_avg_cart_pos",
#     "up_purchase_rate", "up_orders_since_last", "up_share", "up_reorder_density",
#     # User
#     "u_total_orders", "u_total_items", "u_avg_days_between_orders",
#     "u_avg_basket_size", "u_unique_products", "u_reorder_ratio", "u_diversity",
#     # Product
#     "p_purchase_count", "p_reorder_rate", "p_unique_buyers",
#     "p_avg_cart_pos", "p_loyalty"
# ]

# X = final_df[FEATURE_COLS].copy()
# y = final_df["target"].copy()

# # Safe sample — cap at 1M rows
# n_sample = min(1_000_0, len(X))
# idx      = X.sample(n=n_sample, random_state=42).index
# X, y     = X.loc[idx], y.loc[idx]

# print("X shape:", X.shape)
# print("Class balance after filter:")
# print(y.value_counts(normalize=True).round(3))

In [31]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# print("X_train:", X_train.shape, "| X_test:", X_test.shape)
# print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))

In [32]:
# # Apply SMOTE only on train set
# smote = SMOTE(random_state=42)
# X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# print("After SMOTE — train class balance:")
# print(pd.Series(y_train_bal).value_counts())

# # Scale for Logistic Regression
# scaler       = StandardScaler()
# X_train_sc   = scaler.fit_transform(X_train_bal)
# X_test_sc    = scaler.transform(X_test)

---
## 9. Helper: Evaluate Any Model

In [1]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, threshold=0.5):
    """Train, predict, print metrics, return probs."""
    model.fit(X_tr, y_tr)
    probs  = model.predict_proba(X_te)[:, 1]
    preds  = (probs >= threshold).astype(int)

    print(f"\n{'='*60}")
    print(f"  MODEL: {name}  |  threshold={threshold}")
    print(f"{'='*60}")
    print(classification_report(y_te, preds, digits=3))
    print("Confusion Matrix:")
    print(confusion_matrix(y_te, preds))
    print(f"ROC-AUC: {roc_auc_score(y_te, probs):.4f}")
    return probs


def best_threshold(probs, y_true, metric="f1"):
    """Find threshold that maximises F1 for class 1."""
    best_t, best_score = 0.5, 0
    for t in np.arange(0.10, 0.90, 0.02):
        pred  = (probs >= t).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_score:
            best_score, best_t = score, t
    return round(best_t, 2), round(best_score, 4)

    


print("Helper functions defined.")

Helper functions defined.


## 12. Model 3 — XGBoost

In [2]:
# scale_pos_weight compensates for imbalance instead of SMOTE for XGBoost
neg_count  = int((y_train == 0).sum())
pos_count  = int((y_train == 1).sum())
scale_w    = neg_count / pos_count
print(f"scale_pos_weight = {scale_w:.2f}")

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_w,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
# XGBoost handles imbalance with scale_pos_weight — train on original (not SMOTE)
xgb_probs = evaluate("XGBoost", xgb, X_train, y_train, X_test, y_test)

xgb_best_t, xgb_best_f1 = best_threshold(xgb_probs, y_test)
print(f"\nBest threshold: {xgb_best_t}  →  F1 = {xgb_best_f1}")
xgb_probs_final = evaluate("XGBoost (tuned)", xgb, X_train, y_train, X_test, y_test, threshold=xgb_best_t)

NameError: name 'y_train' is not defined

---
## 14. Model Comparison Summary

In [35]:
def summary_row(name, probs, y_true, threshold):
    pred = (probs >= threshold).astype(int)
    return {
        "Model":     name,
        "Threshold": threshold,
        "Precision": round(precision_score(y_true, pred, zero_division=0), 3),
        "Recall":    round(recall_score(y_true, pred, zero_division=0), 3),
        "F1":        round(f1_score(y_true, pred, zero_division=0), 3),
        "ROC-AUC":   round(roc_auc_score(y_true, probs), 4)
    }

summary = pd.DataFrame([
  
    summary_row("XGBoost",             xgb_probs,        y_test, xgb_best_t),
    
])

print(summary.to_string(index=False))
print(f"\nBest model by F1: {summary.loc[summary['F1'].idxmax(), 'Model']}")

  Model  Threshold  Precision  Recall    F1  ROC-AUC
XGBoost       0.72      0.383   0.493 0.431   0.8293

Best model by F1: XGBoost


## 15. Feature Importance (XGBoost)

In [36]:
imp = pd.DataFrame({
    "Feature":    FEATURE_COLS,
    "Importance": xgb.feature_importances_
}).sort_values("Importance", ascending=False)

print("XGBoost Feature Importances:")
print(imp.to_string(index=False))

XGBoost Feature Importances:
                  Feature  Importance
     up_orders_since_last    0.327050
          up_times_bought    0.321093
         up_purchase_rate    0.123093
       up_reorder_density    0.038285
           p_reorder_rate    0.031990
              u_diversity    0.015951
            up_last_order    0.014662
                p_loyalty    0.014537
           up_first_order    0.013431
          u_reorder_ratio    0.012712
           u_total_orders    0.011952
         p_purchase_count    0.011796
          p_unique_buyers    0.010425
u_avg_days_between_orders    0.008194
           p_avg_cart_pos    0.008070
                 up_share    0.007849
        u_avg_basket_size    0.007774
        u_unique_products    0.007428
            u_total_items    0.007110
          up_avg_cart_pos    0.006596


In [37]:
# FINAL GOAL: recommend product names for each user

best_threshold = xgb_best_t   # your best threshold, example 0.70

# Predict reorder probability for all candidate user-product pairs
all_X = final_df[FEATURE_COLS].copy()
all_probs = xgb.predict_proba(all_X)[:, 1]

recommendations = final_df[["user_id", "product_id"]].copy()
recommendations["reorder_probability"] = all_probs

# Keep products above threshold
recommendations = recommendations[
    recommendations["reorder_probability"] >= best_threshold
]

# Add product names
recommendations = recommendations.merge(
    products[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

# Sort highest probability first
recommendations = recommendations.sort_values(
    ["user_id", "reorder_probability"],
    ascending=[True, False]
)

recommendations.head(20)

,user_id,product_id,reorder_probability,product_name
0,1,196,0.979738,Soda
2,1,12427,0.976892,Original Beef Jerky
1,1,10258,0.971343,Pistachios
4,1,25133,0.966467,Organic String Cheese
8,1,46149,0.942363,Zero Calorie Cola
6,1,38928,0.856584,0% Greek Strained Yogurt
7,1,39657,0.849758,Milk Chocolate Almonds
5,1,35951,0.824713,Organic Unsweetened Almond Milk
9,1,49235,0.811205,Organic Half & Half
3,1,13032,0.803475,Cinnamon Toast Crunch


In [38]:
def predict_products_for_user(user_id, top_n=5):
    user_recs = recommendations[recommendations["user_id"] == user_id].copy()

    if user_recs.empty:
        print(f"User {user_id}")
        print("\nPredicted products:\n")
        print("No products predicted for reorder.")
        return

    user_recs = user_recs.sort_values(
        by="reorder_probability",
        ascending=False
    ).head(top_n)

    product_names = user_recs["product_name"].tolist()

    print(f"User {user_id}")
    print("\nPredicted products:\n")
    print(" , ".join(product_names))

In [39]:
user_id = int(input("Enter user id: "))

predict_products_for_user(user_id, top_n=5)

Enter user id:  1


User 1

Predicted products:

Soda , Original Beef Jerky , Pistachios , Organic String Cheese , Zero Calorie Cola
